In [10]:
import os
import pickle
import random
from pathlib import Path
from sklearn.model_selection import train_test_split


class SimpleRAFDBDataset:
    def __init__(self, root_dir, split='train'):
        self.root_dir = Path(root_dir)
        self.split = split
        self.images = []
        self.labels = []
        self._load_data()
    
    def _load_data(self):
        data_dir = self.root_dir / self.split
        
        for class_folder in sorted(data_dir.iterdir()):
            if not class_folder.is_dir():
                continue
            try:
                class_idx = int(class_folder.name)
                if class_idx not in range(1, 8):
                    continue
                label = class_idx - 1
                image_files = list(class_folder.glob('*.jpg')) + list(class_folder.glob('*.png'))
                for img_path in image_files:
                    self.images.append(str(img_path))
                    self.labels.append(label)
            except ValueError:
                continue
        
        print(f"Загружено {len(self.images)} изображений")
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]


def create_fixed_split(root_dir, val_split=0.15, random_seed=42, save_path="dataset_splits.pkl"):
    full_train = SimpleRAFDBDataset(root_dir, split='train')
    
    all_indices = list(range(len(full_train)))
    all_labels = full_train.labels
    
    train_indices, val_indices = train_test_split(
        all_indices,
        test_size=val_split,
        random_state=random_seed,
        stratify=all_labels
    )
    
    test_dataset = SimpleRAFDBDataset(root_dir, split='test')
    test_indices = list(range(len(test_dataset)))
    
    splits = {
        'train_indices': train_indices,
        'val_indices': val_indices,
        'test_indices': test_indices,
        'train_labels': [full_train.labels[i] for i in train_indices],
        'val_labels': [full_train.labels[i] for i in val_indices],
        'test_labels': test_dataset.labels,
        'random_seed': random_seed,
        'val_split': val_split
    }
    
    with open(save_path, 'wb') as f:
        pickle.dump(splits, f)
    
    print(f"\nРазбиение сохранено в {save_path}")
    print(f" Train: {len(train_indices)}  Val: {len(val_indices)}  Test: {len(test_indices)}")
    
    emotion_names = {0: 'Surprise', 1: 'Fear', 2: 'Disgust', 
                     3: 'Happiness', 4: 'Sadness', 5: 'Anger', 6: 'Neutral'}
    
    print("\nРаспределение по классам:")
    for class_idx in range(7):
        name = emotion_names[class_idx]
        train_count = splits['train_labels'].count(class_idx)
        val_count = splits['val_labels'].count(class_idx)
        test_count = splits['test_labels'].count(class_idx)
        print(f" {name:9} train {train_count:4d}  val {val_count:3d}  test {test_count:3d}")
    
    return splits


def load_fixed_split(splits_path="dataset_splits.pkl"):
    
    with open(splits_path, 'rb') as f:
        splits = pickle.load(f)
    
    print(f" Train: {len(splits['train_indices']):5d}")
    print(f" Val:   {len(splits['val_indices']):5d}")
    print(f" Test:  {len(splits['test_indices']):5d}")
    
    return splits


if __name__ == "__main__":
    RAFDB_ROOT = r"D:\НИР\RAF-DB\DATASET"
    
    splits = create_fixed_split(
        root_dir=RAFDB_ROOT,
        val_split=0.15,
        random_seed=42,
        save_path="dataset_splits.pkl"
    )
    
    loaded_splits = load_fixed_split("dataset_splits.pkl")

Загружено 12271 изображений
Загружено 3068 изображений

Разбиение сохранено в dataset_splits.pkl
 Train: 10430  Val: 1841  Test: 3068

Распределение по классам:
 Surprise  train 1097  val 193  test 329
 Fear      train  239  val  42  test  74
 Disgust   train  609  val 108  test 160
 Happiness train 4056  val 716  test 1185
 Sadness   train 1685  val 297  test 478
 Anger     train  599  val 106  test 162
 Neutral   train 2145  val 379  test 680
 Train: 10430
 Val:    1841
 Test:   3068


In [18]:
from pathlib import Path
import pickle

def collect_paths(root):
    root = Path(root)
    paths = []
    for p in sorted(root.rglob("*")):
        if p.suffix.lower() in [".jpg", ".png"]:
            paths.append(str(p.resolve()))
    return paths


def build_split_sets(splits, train_dataset, root_dir):
    root_dir = Path(root_dir)

    train_paths = [str(Path(train_dataset.images[i]).resolve())
                   for i in splits["train_indices"]]

    val_paths = [str(Path(train_dataset.images[i]).resolve())
                 for i in splits["val_indices"]]

    test_paths = collect_paths(root_dir / "test")

    return set(train_paths), set(val_paths), set(test_paths)


def audit(root_dir, splits_path, train_dataset):
    with open(splits_path, "rb") as f:
        splits = pickle.load(f)

    train_set, val_set, test_set = build_split_sets(splits, train_dataset, root_dir)

    print("Train и Val :", len(train_set & val_set))
    print("Train и Test:", len(train_set & test_set))
    print("Val и Test  :", len(val_set & test_set))


RAFDB_ROOT = r"D:/НИР/RAF-DB/DATASET"
SPLITS_PATH = "D:\НИР\dataset_splits.pkl"

train_dataset = SimpleRAFDBDataset(RAFDB_ROOT, split="train")

audit(
    root_dir=RAFDB_ROOT,
    splits_path=SPLITS_PATH,
    train_dataset=train_dataset
)

Загружено 12271 изображений
Train и Val : 0
Train и Test: 0
Val и Test  : 0
